In [3]:
'''
Current System:
Question -> FAISS -> Top 5

New System that we would build:
Questions -> BM25 | FAISS -> Top 20  -> RRF -> Final top results
'''

'\nCurrent System:\nQuestion -> FAISS -> Top 5\n\nNew System that we would build:\nQuestions -> BM25 | FAISS -> Top 20  -> RRF -> Final top results\n'

In [4]:
# Install Libraries
!pip install -q \
langchain \
langchain-community \
langchain-huggingface \
langchain-text-splitters \
faiss-cpu \
rank-bm25 \
sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [5]:
# Imports
import numpy as np
from pathlib import Path
from rank_bm25 import BM25Okapi
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

/tmp/ipykernel_2108/2836005086.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
ROOT_DIR = Path("/content/drive/MyDrive/legal_ai/CUAD_v1")
TXT_DIR = ROOT_DIR / "full_contract_txt"
FAISS_DIR = Path("/content/drive/MyDrive/legal_ai/vectorstores/legal_faiss")

print(TXT_DIR.exists())
print(FAISS_DIR.exists())

True
True


In [8]:
#  lets load contracts
from langchain_core.documents import Document

documents = []

for txt_file in TXT_DIR.rglob("*.txt"):

    with open(
        txt_file,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        text = f.read()

    filename = txt_file.stem

    contract_type = filename.split("_")[-1]

    documents.append(
        Document(
            page_content=text,
            metadata={
                "source": txt_file.name,
                "contract_type": contract_type
            }
        )
    )

print("Documents Loaded:", len(documents))

Documents Loaded: 510


In [9]:
print(documents[0].page_content)


Confidential treatment has been requested with respect to portions of this agreement as indicated by "[***]" and such confidential portions have been deleted and filed separately with the Securities and Exchange Commission pursuant to Rule 24b-2 of the Securities Exchange Act of 1934, as amended. Amendment #3 to the Manufacturing Agreement This Amendment #3 to the Manufacturing Agreement (this "Amendment #3") is made effective as of December 21, 2017 ("Amendment Effective Date"), by and between ADMA BioManufacturing, LLC, a Delaware limited liability company, having a place of business at 5800 Park of Commerce Boulevard NW, Boca Raton, Florida 33487 USA ("ADMA") and Sanofi Pasteur S.A., a company existing and organized under the laws of France ("Sanofi Pasteur"), having its registered head office at 14, espace Henry Vallee, 69007, Lyon, France. WHEREAS, ADMA (as successor-in-interest to Biotest Pharmaceuticals Corporation ("BPC") and Sanofi Pasteur are parties to that certain Manufactu

In [10]:
print(documents[10].metadata)
print(documents[10].metadata.items())

{'source': 'AimmuneTherapeuticsInc_20200205_8-K_EX-10.3_11967170_EX-10.3_Development Agreement.txt', 'contract_type': 'Development Agreement'}
dict_items([('source', 'AimmuneTherapeuticsInc_20200205_8-K_EX-10.3_11967170_EX-10.3_Development Agreement.txt'), ('contract_type', 'Development Agreement')])


In [11]:
contract_types = []

for doc in documents:
    contract_types.append(
        doc.metadata["contract_type"]
    )

print("Unique Types:", len(set(contract_types)))

for c in sorted(set(contract_types))[:50]:
    print(c)

Unique Types: 370
 Agency Agreement
 Investment Distribution Agreement
 MARKETING AGREEMENT
 Reseller Agreement
 Sales-Purchase Agreement1
 Sales-Purchase Agreement2
 Service Agreement
 Supply Agreement
 WILCOX ENTERPRISES, INC.
1994-EX-10.47-EXCLUSIVE DISTRIBUTOR AGREEMENT
1996-EX-1.1-AGENCY AGREEMENT
1996-EX-10.12-ENDORSEMENT AGREEMENT
1996-EX-10.4-SPONSORSHIP AGREEMENT
1996-EX-10.4-TRANSPORTATION SERVICES AGREEMENT
1997-EX-10.11-SPONSORSHIP AGREEMENT
1997-EX-10.16-SPONSORSHIP AGREEMENT
1997-EX-10.2-10-ENDORSEMENT AGREEMENT
1997-EX-10.28-ENDORSEMENT AGREEMENT
1997-EX-10.46-WEB HOSTING AGREEMENT
1997-EX-10.47-SPONSORSHIP AGREEMENT
1997-EX-4-AFFILIATE AGREEMENT
1997-EX-99-FRANCHISE AGREEMENT
1998-EX-10-FRANCHISE AGREEMENT
1998-EX-10-OUTSOURCING AGREEMENT
1998-EX-10-SPONSORSHIP AGREEMENT
1998-EX-10.13-PROMOTION AGREEMENT
1998-EX-10.14-OUTSOURCING AGREEMENT
1998-EX-10.14-SOFTWARE HOSTING AGREEMENT
1998-EX-10.15-SPONSORSHIP AGREEMENT
1998-EX-10.16-SPONSORSHIP AGREEMENT
1998-EX-10.17-SPONS

In [12]:
'''our current contract metadata is noisy, and if we build filtering and risk analysis
by this we will get 370 categories which is useless so we will try to optimise the
contract extraction via regular expressions.  '''

# Legal Contract Taxonomy
def extract_contract_type(filename):

    filename = filename.lower()

    contract_patterns = [

        "manufacturing agreement",
        "license agreement",
        "affiliate agreement",
        "consulting agreement",
        "distribution agreement",
        "development agreement",
        "supply agreement",
        "joint venture agreement",
        "strategic alliance agreement",
        "outsourcing agreement",
        "sponsorship agreement",
        "maintenance agreement",
        "service agreement",
        "agency agreement",
        "promotion agreement",
        "hosting agreement",
        "franchise agreement",
        "endorsement agreement",
        "cooperation agreement",
        "transportation agreement"
    ]

    for pattern in contract_patterns:

        if pattern in filename:

            return pattern.title()

    return "Other"

In [13]:
#  lets load contracts
from langchain_core.documents import Document

documents = []

for txt_file in TXT_DIR.rglob("*.txt"):

    with open(
        txt_file,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        text = f.read()

    filename = txt_file.stem

    contract_type = extract_contract_type(filename)

    documents.append(
        Document(
            page_content=text,
            metadata={
                "source": txt_file.name,
                "contract_type": contract_type
            }
        )
    )

print("Documents Loaded:", len(documents))

Documents Loaded: 510


In [14]:
contract_types = []

for doc in documents:

    contract_types.append(
        doc.metadata["contract_type"]
    )


In [15]:
unique_types = sorted(
    set(contract_types)
)

print("Total Unique Types:", len(unique_types))
print()

for contract_type in unique_types:
    print(contract_type)

Total Unique Types: 21

Affiliate Agreement
Agency Agreement
Consulting Agreement
Cooperation Agreement
Development Agreement
Distribution Agreement
Endorsement Agreement
Franchise Agreement
Hosting Agreement
Joint Venture Agreement
License Agreement
Maintenance Agreement
Manufacturing Agreement
Other
Outsourcing Agreement
Promotion Agreement
Service Agreement
Sponsorship Agreement
Strategic Alliance Agreement
Supply Agreement
Transportation Agreement


In [16]:
from collections import Counter

type_counts = Counter(contract_types)

print("Contract Type Counts")
print("=" * 50)

for contract_type, count in sorted(
    type_counts.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{contract_type:<30} {count}")

Contract Type Counts
Other                          150
License Agreement              34
Maintenance Agreement          32
Strategic Alliance Agreement   32
Sponsorship Agreement          30
Development Agreement          28
Endorsement Agreement          24
Supply Agreement               21
Outsourcing Agreement          18
Hosting Agreement              18
Service Agreement              15
Cooperation Agreement          15
Franchise Agreement            15
Manufacturing Agreement        14
Agency Agreement               14
Consulting Agreement           11
Promotion Agreement            11
Affiliate Agreement            10
Joint Venture Agreement        9
Transportation Agreement       6
Distribution Agreement         3


In [17]:
other_examples = []

for doc in documents:

    if doc.metadata["contract_type"] == "Other":

        other_examples.append(
            doc.metadata["source"]
        )

print("Total Others:", len(other_examples))
print()

for file in other_examples[:50]:
    print(file)

Total Others: 150

BERKELEYLIGHTS,INC_06_26_2020-EX-10.12-COLLABORATION AGREEMENT.txt
ARMSTRONGFLOORING,INC_01_07_2019-EX-10.2-INTELLECTUAL PROPERTY AGREEMENT.txt
AULAMERICANUNITTRUST_04_24_2020-EX-99.8.77-SERVICING AGREEMENT.txt
Array BioPharma Inc. - LICENSE, DEVELOPMENT AND COMMERCIALIZATION AGREEMENT.txt
ASIANDRAGONGROUPINC_08_11_2005-EX-10.5-Reseller Agreement.txt
ANIXABIOSCIENCESINC_06_09_2020-EX-10.1-COLLABORATION AGREEMENT.txt
ACCURAYINC_09_01_2010-EX-10.31-DISTRIBUTOR AGREEMENT.txt
ADIANUTRITION,INC_04_01_2005-EX-10.D2-RESELLER AGREEMENT.txt
AIRSPANNETWORKSINC_04_11_2000-EX-10.5-Distributor Agreement.txt
ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT.txt
ArmstrongFlooringInc_20190107_8-K_EX-10.2_11471795_EX-10.2_Intellectual Property Agreement.txt
BABCOCK_WILCOXENTERPRISES,INC_08_04_2015-EX-10.17-INTELLECTUAL PROPERTY AGREEMENT between THE BABCOCK _ WILCOX COMPANY and BABCOCK _ WILCOX ENTERPRISES, INC..txt
ATENTOSA_07_06_2020-EX-99.1-JOINT FILING AGREEMENT.txt
CERES,INC_01_2

In [18]:
#  Using llms to extract contract type
sample_docs = documents[:30]

for i, doc in enumerate(sample_docs):

    print("="*80)
    print(f"DOCUMENT {i+1}")

    print(doc.metadata["source"])

    print(doc.page_content[:500])

DOCUMENT 1
ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
Confidential treatment has been requested with respect to portions of this agreement as indicated by "[***]" and such confidential portions have been deleted and filed separately with the Securities and Exchange Commission pursuant to Rule 24b-2 of the Securities Exchange Act of 1934, as amended. Amendment #3 to the Manufacturing Agreement This Amendment #3 to the Manufacturing Agreement (this "Amendment #3") is made effective as of December 21, 2017 ("Amendment Effective Date"), by and between 
DOCUMENT 2
ALLIANCEBANCORPINCOFPENNSYLVANIA_10_18_2006-EX-1.2-AGENCY AGREEMENT.txt
Exhibit 1.2

Up to 2,445,223 Shares

(subject to increase to up to 2,812,006 shares in the event of an increase in the pro forma market value of the Company's Common Stock)

Alliance Bancorp, Inc. of Pennsylvania (a federal stock holding company)

Common Stock (par value $.01 per share)

AGENCY AGREEMENT

November ___, 2006

SAN

In [19]:
sample_docs = documents[:30]

In [20]:
# Prompt classify
from langchain_core.prompts import ChatPromptTemplate

classification_prompt = ChatPromptTemplate.from_template(
"""
You are a legal contract classification expert.

Your task is to classify the contract into EXACTLY ONE category from the list below.

ALLOWED CATEGORIES:

- Manufacturing Agreement
- License Agreement
- Affiliate Agreement
- Agency Agreement
- Consulting Agreement
- Distribution Agreement
- Development Agreement
- Supply Agreement
- Joint Venture Agreement
- Strategic Alliance Agreement
- Outsourcing Agreement
- Sponsorship Agreement
- Maintenance Agreement
- Service Agreement
- Promotion Agreement
- Hosting Agreement
- Franchise Agreement
- Endorsement Agreement
- Cooperation Agreement
- Transportation Agreement
- Collaboration Agreement
- Intellectual Property Agreement
- Marketing Agreement
- Reseller Agreement
- Servicing Agreement
- Other

RULES:

1. Return ONLY one category.
2. Do NOT explain your answer.
3. Do NOT generate sentences.
4. If multiple categories appear, choose the PRIMARY purpose of the agreement.
5. If none apply, return "Other".

CONTRACT:

{contract_text}

CATEGORY:
"""
)

In [21]:
import os
os.environ["OPENAI_API_KEY"] ="bedrock-api-key-YmVkcm9jay5hbWF6b25hd3MuY29tLz9BY3Rpb249Q2FsbFdpdGhCZWFyZXJUb2tlbiZYLUFtei1BbGdvcml0aG09QVdTNC1ITUFDLVNIQTI1NiZYLUFtei1DcmVkZW50aWFsPUFTSUFYNjZWV1o0S1ZXWFY2MklJJTJGMjAyNjA2MjElMkZldS1ub3J0aC0xJTJGYmVkcm9jayUyRmF3czRfcmVxdWVzdCZYLUFtei1EYXRlPTIwMjYwNjIxVDEyMTY1OVomWC1BbXotRXhwaXJlcz00MzIwMCZYLUFtei1TZWN1cml0eS1Ub2tlbj1JUW9KYjNKcFoybHVYMlZqRUNRYUNtVjFMVzV2Y25Sb0xURWlSekJGQWlCT3JPZFp3RXllbHE4emVOY0pxNmpaajQ2eG1MVGpYUkNUbm5URkg4dlhuUUloQU1OVnc4Qzk0cWU2VndDTiUyRmsyJTJCNlRodFZzekFKYzBzY0k3eVBwS2lSWmVtS3JJRENPMyUyRiUyRiUyRiUyRiUyRiUyRiUyRiUyRiUyRiUyRndFUUFCb01OVFEzTlRFNU5qUTNOVEE1SWd4SENBZ3NuJTJCQlRVeDdaQm5FcWhnUGJCZVolMkI0cFVTY0NSQ0FRblVmR0llWHV2a3hlR0Jmbk9UVzAlMkJLMHV6OFRVJTJCeGdGYSUyQnVta2U3UUUxV3IxTU5GJTJGZTdaZ3JxMllySzlyeFElMkY5R0Fyc2hxTTUlMkJXQjZJcmRzTmVXbG15ZkZzZnVyYWk4ZjhBOSUyQktEUEdrR1o5MEk5cHFrU09veWtzWVZqSWU1d0VKNTJFOVZzUXQ5SVRUdlVlYk9LUHM1NU9DMTczNXZtY2VaWkpWNndOOFlaR0xVeHB1OHpzNnVCd2R2eklUVFNRdHN3eUx1eWRCOXYxcTV3cU51TWxrUW02SlQ1MVhwckJycFZNQTJkZTFYWVFhNzY4RHVWNG5iYU81ZXRaWUFvJTJCbG1MdXJwQzFxa1h2dGdadVBxMm1pZmVKcyUyRmxMeHROcldrJTJCOThVJTJCNnRscFFXWVBIQ3FIOEZYOWJ3JTJGTXk1eUF0N3A3VldsbGFGZmV0VyUyRkZvTjF2eGo4ZG05ZjJKdjRrV0VHR3ZkdFVNRlpscncxQ1IlMkJWN2VxUkxFdWRoMXdsYXQ1RGJ3THRyTExOM2U2aGJtd2UyeDRjdGxHYkdaTVlvTFlXNGZ5UzhjOUNobVFYJTJCTmVrOEJGSDM3MmRxMEpkRFBEaVFBUjdYMG55VnNTa2JNMzZBNUZNVmhtNDMwdk1RZ0N1NlJJTFlsR0xHVDZQdVA2NkhkRHpEWXdzNnpmMFFZNjNnTGpwTFNFWTBCUGQwdndUdGJLdUFvJTJCMzVlT2FmSmZBU1hJMkt6ZElZaWhXJTJGdmVGbmJXZlljdXlIN3VKbkhwNSUyQkptVTdDYVl6OSUyRnZFcU9iWGtEJTJGek4wME5iWFF2ekZTdG84a041cE82dWh6WlBJNm02JTJGSk9aS1RFMHJFMlJFRk9paUs5SnJWNWxUeG9admFlalBqaFZuaERjTnZERlR4TVpQbDFrUmo5QmoyOERDblNsMGZ5NWtib2dlNGxPTmV4c0FzVTRIZklid1VoaDM5ZUJCTDFMeENjcHRFdWVxTzZxQzVjb3NobmVMS0NKMlNhSDJXNCUyRjdhTjRXcVZES2tqaUgzNlB0am5wdHB4cm5iVzFDN0M0TWc2anZ5aFpXQnlJSFQxRmdZTE5YRUJZdlFPbmNTSTQ1Y0txWUdBUjY3QkxrYTNDdDdPZUFKOGVXRSUyQnFKSlpnOFIlMkIyVm9xSmU3Z1N0OUp6TnU4eFBUcGNKSnhrWFpVcTV1VTJGVjNmQ3I3NkZDbU0lMkJWQW9IdGJsc0xabzY1NTNYSmtFcXVLYnl1aFBEcUFkajY2WkJxaFhpdWRqZGIwUk1MTzYlMkZMNGNzOWYzdm1uejJuVXolMkYlMkY0ZWhhbDFWeGclM0QlM0QmWC1BbXotU2lnbmF0dXJlPWNiMTg4MWM0ZWUwYjc1MjQ3MTdjODc3OWNkNzQ1NzUxYmIzOTM4MzVmZWIzMmU1YWI4ZjAwZTRmYmNhZTlhODgmWC1BbXotU2lnbmVkSGVhZGVycz1ob3N0JlZlcnNpb249MQ=="
os.environ["OPENAI_BASE_URL"]= "https://bedrock-mantle.eu-north-1.api.aws/v1"


In [22]:
!pip install langchain-openai
from openai import OpenAI
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai.gpt-oss-120b",
    temperature=0
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 11.6 MB/s eta 0:00:00


In [23]:
response = llm.invoke("What is a Black Hole?")
print(response.content[:100])

**A black hole is a region of space where gravity is so strong that nothing—not even light—can escap


In [24]:
def classify_contract(contract_text):

    chain = classification_prompt | llm

    result = chain.invoke(
        {
            "contract_text": contract_text[:2000]
        }
    )

    return result.content.strip()

In [25]:
for doc in sample_docs:

    prediction = classify_contract(
        doc.page_content
    )

    print()
    print(doc.metadata["source"])
    print("Predicted:", prediction)


ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt
Predicted: Manufacturing Agreement

ALLIANCEBANCORPINCOFPENNSYLVANIA_10_18_2006-EX-1.2-AGENCY AGREEMENT.txt
Predicted: Agency Agreement

BELLRINGBRANDS,INC_02_07_2020-EX-10.18-MASTER SUPPLY AGREEMENT.txt
Predicted: Supply Agreement

AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.txt
Predicted: License Agreement

ArconicRolledProductsCorp_20191217_10-12B_EX-2.7_11923804_EX-2.7_Trademark License Agreement.txt
Predicted: License Agreement

AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1.txt
Predicted: Maintenance Agreement

ATMOSENERGYCORP_11_22_2002-EX-10.17-TRANSPORTATION SERVICE AGREEMENT.txt
Predicted: Transportation Agreement

AFSALABANCORPINC_08_01_1996-EX-1.1-AGENCY AGREEMENT.txt
Predicted: Agency Agreement

ArcaUsTreasuryFund_20200207_N-2_EX-99.K5_11971930_EX-99.K5_Development Agreement.txt
Predicted: Development Agreement

AlliedEsportsE

In [26]:
# metadata base
records = []

for doc in documents:

    prediction = classify_contract(
        doc.page_content[:1200]
    )

    records.append({
        "source": doc.metadata["source"],
        "contract_type": prediction
    })

In [27]:
import pandas as pd

contract_metadata = pd.DataFrame(records)

contract_metadata.head(10)

,source,contract_type
0,"ADMA BioManufacturing, LLC - Amendment #3 to ...",Manufacturing Agreement
1,ALLIANCEBANCORPINCOFPENNSYLVANIA_10_18_2006-EX...,Agency Agreement
2,"BELLRINGBRANDS,INC_02_07_2020-EX-10.18-MASTER ...",Supply Agreement
3,AlliedEsportsEntertainmentInc_20190815_8-K_EX-...,License Agreement
4,ArconicRolledProductsCorp_20191217_10-12B_EX-2...,License Agreement
5,AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_M...,Maintenance Agreement
6,ATMOSENERGYCORP_11_22_2002-EX-10.17-TRANSPORTA...,Transportation Agreement
7,AFSALABANCORPINC_08_01_1996-EX-1.1-AGENCY AGRE...,Agency Agreement
8,ArcaUsTreasuryFund_20200207_N-2_EX-99.K5_11971...,Development Agreement
9,AlliedEsportsEntertainmentInc_20190815_8-K_EX-...,Sponsorship Agreement


In [28]:
from pathlib import Path

METADATA_DIR = Path(
    "/content/drive/MyDrive/legal_ai/metadata"
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [29]:
contract_metadata.to_csv(
    METADATA_DIR / "contract_type_mapping.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [30]:
from tqdm import tqdm
records = []

for doc in tqdm(documents):

    prediction = classify_contract(
        doc.page_content[:1200]
    )

    records.append({
        "source": doc.metadata["source"],
        "contract_type": prediction
    })

100%|██████████| 510/510 [05:40<00:00,  1.50it/s]


In [31]:
import pandas as pd

contract_metadata = pd.DataFrame(records)

print(contract_metadata.shape)

contract_metadata.head()

(510, 2)


,source,contract_type
0,"ADMA BioManufacturing, LLC - Amendment #3 to ...",Manufacturing Agreement
1,ALLIANCEBANCORPINCOFPENNSYLVANIA_10_18_2006-EX...,Agency Agreement
2,"BELLRINGBRANDS,INC_02_07_2020-EX-10.18-MASTER ...",Supply Agreement
3,AlliedEsportsEntertainmentInc_20190815_8-K_EX-...,License Agreement
4,ArconicRolledProductsCorp_20191217_10-12B_EX-2...,License Agreement


In [32]:
type_counts = (
    contract_metadata["contract_type"]
    .value_counts()
)

print(type_counts)

contract_type
License Agreement                  53
Distribution Agreement             37
Strategic Alliance Agreement       33
Sponsorship Agreement              31
Endorsement Agreement              25
Maintenance Agreement              24
Development Agreement              22
Service Agreement                  21
Marketing Agreement                21
Supply Agreement                   19
Outsourcing Agreement              19
Cooperation Agreement              19
Manufacturing Agreement            18
Other                              18
Intellectual Property Agreement    17
Agency Agreement                   17
Collaboration Agreement            15
Franchise Agreement                15
Consulting Agreement               14
Transportation Agreement           13
Hosting Agreement                  13
Reseller Agreement                 12
Promotion Agreement                11
Joint Venture Agreement            10
Affiliate Agreement                10
Servicing Agreement                 

In [33]:
from pathlib import Path
PROJECT_DIR = Path("/content/drive/MyDrive/legal_ai")
METADATA_DIR = PROJECT_DIR / "metadata"

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [34]:
contract_metadata.to_csv(
    METADATA_DIR / "contract_type_mapping.csv",
    index=False
)

print("Metadata Saved")

Metadata Saved


In [35]:
contract_type_lookup = dict(
    zip(
        contract_metadata["source"],
        contract_metadata["contract_type"]
    )
)

In [36]:
for doc in documents:

    source = doc.metadata["source"]

    doc.metadata["contract_type"] = (
        contract_type_lookup[source]
    )

In [37]:
print(documents[150].metadata)

{'source': 'ImperialGardenResortInc_20161028_DRS (on F-1)_EX-10.13_9963189_EX-10.13_Outsourcing Agreement.txt', 'contract_type': 'Outsourcing Agreement'}


In [38]:
import pickle

with open(
    METADATA_DIR / "documents_with_metadata.pkl",
    "wb"
) as f:

    pickle.dump(documents, f)

In [39]:
METADATA_DIR = Path(
    "/content/drive/MyDrive/legal_ai/metadata"
)

for file in METADATA_DIR.iterdir():
    print(file.name)

contract_type_mapping.csv
documents_with_metadata.pkl


Chunking and Embeddings

In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(
    documents
)

print("Total Chunks:", len(chunks))

Total Chunks: 39025


In [41]:
print(chunks[0].metadata)

{'source': 'ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement .txt', 'contract_type': 'Manufacturing Agreement'}


In [42]:
chunk_lengths = [
    len(chunk.page_content)
    for chunk in chunks
]

print("Min:", min(chunk_lengths))
print("Max:", max(chunk_lengths))
print("Average:",
    round(sum(chunk_lengths)/len(chunk_lengths),2)
)

Min: 1
Max: 1000
Average: 762.97


In [43]:
small_chunks = [
    chunk
    for chunk in chunks
    if len(chunk.page_content) < 50
]

print("Small Chunks:", len(small_chunks))

Small Chunks: 1207


In [44]:
for chunk in small_chunks[:10]:
    print("="*50)
    print(len(chunk.page_content))
    print(repr(chunk.page_content))

46
'BPC Initials ___ Sanofi Pasteur Initials ___ 1'
46
'BPC Initials ___ Sanofi Pasteur Initials ___ 2'
46
'BPC Initials ___ Sanofi Pasteur Initials ___ 3'
46
'BPC Initials ___ Sanofi Pasteur Initials ___ 4'
32
'Source: AZUL SA, F-1/A, 3/3/2017'
32
'Source: AZUL SA, F-1/A, 3/3/2017'
32
'Source: AZUL SA, F-1/A, 3/3/2017'
32
'Source: AZUL SA, F-1/A, 3/3/2017'
32
'Source: AZUL SA, F-1/A, 3/3/2017'
32
'Source: AZUL SA, F-1/A, 3/3/2017'


In [45]:
print("Before Cleaning:", len(chunks))

chunks = [
    chunk
    for chunk in chunks
    if len(chunk.page_content.strip()) >= 50
]

print("After Cleaning:", len(chunks))

Before Cleaning: 39025
After Cleaning: 37818


In [46]:
chunk_lengths = [
    len(chunk.page_content)
    for chunk in chunks
]

print("Min:", min(chunk_lengths))
print("Max:", max(chunk_lengths))
print(
    "Average:",
    round(sum(chunk_lengths)/len(chunk_lengths),2)
)

Min: 50
Max: 1000
Average: 786.51


In [47]:
print(chunks[5550].metadata)

{'source': 'EbixInc_20010515_10-Q_EX-10.3_4049767_EX-10.3_Co-Branding Agreement.txt', 'contract_type': 'Marketing Agreement'}


In [48]:
PROJECT_DIR = Path("/content/drive/MyDrive/legal_ai")
METADATA_DIR = PROJECT_DIR / "metadata"
VECTOR_DIR = PROJECT_DIR / "vectorstores"

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VECTOR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project :", PROJECT_DIR)
print("Metadata:", METADATA_DIR)
print("Vectors :", VECTOR_DIR)

Project : /content/drive/MyDrive/legal_ai
Metadata: /content/drive/MyDrive/legal_ai/metadata
Vectors : /content/drive/MyDrive/legal_ai/vectorstores


In [49]:
with open(
    VECTOR_DIR / "chunks.pkl",
    "wb"
) as f:

    pickle.dump(chunks, f)


Embeddings

In [50]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

/tmp/ipykernel_2108/950273920.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

VectorStore FAISS

In [51]:
from langchain_community.vectorstores import FAISS
faiss_db = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [52]:
FAISS_DIR = VECTOR_DIR / "legal_faiss"

faiss_db.save_local(
    str(FAISS_DIR)
)


In [53]:
results = faiss_db.similarity_search(
    "What confidentiality obligations exist?",
    k=5
)

for i, doc in enumerate(results, start=1):

    print("="*80)
    print(f"RESULT {i}")

    print("\nSOURCE:")
    print(doc.metadata["source"])

    print("\nTYPE:")
    print(doc.metadata["contract_type"])

    print("\nTEXT:")
    print(doc.page_content[:500])

RESULT 1

SOURCE:
OFGBANCORP_03_28_2007-EX-10.23-OUTSOURCING AGREEMENT.txt

TYPE:
Outsourcing Agreement

TEXT:
prevent unauthorized access to such materials. Each party shall instruct its employees, agents, and contractors (a) of its confidentiality obligations hereunder and (b) not to attempt to circumvent any such security procedures and devices. Each party's obligation under the preceding sentence may be satisfied by the use of its standard form of confidentiality agreement, if the same reasonably accomplishes the purposes here intended. All such Confidential Information shall be distributed only to p
RESULT 2

SOURCE:
TRICITYBANKSHARESCORP_05_15_1998-EX-10-OUTSOURCING AGREEMENT.txt

TYPE:
Outsourcing Agreement

TEXT:
of its confidentiality obligations hereunder and not to attempt to circumvent any such security procedures and devices.  Each party's obligation under the preceding sentence may be satisfied by the use of its standard form of confidentiality agreement, if the same reas

BM25 Retriever

In [54]:
from langchain_community.retrievers import BM25Retriever
bm25_retriever = BM25Retriever.from_documents(
    chunks
)

In [55]:
bm25_retriever.k = 5

In [56]:
results = bm25_retriever.invoke(
    "governing law"
)

for i, doc in enumerate(results, start=1):

    print("="*80)

    print(doc.metadata["source"])

    print("\n")

    print(doc.page_content[:400])

Array BioPharma Inc. - LICENSE, DEVELOPMENT AND COMMERCIALIZATION AGREEMENT.txt


For the avoidance of doubt, the governing law set forth in Section 18.3 shall not apply to determine any procedural issues. In particular, but without in any way restricting the generality of the foregoing, the Parties agree that the procedural rules of the governing law set forth in Section 18.3 shall not apply with respect to document production or other evidentiary issues, except that all privi
ATMOSENERGYCORP_11_22_2002-EX-10.17-TRANSPORTATION SERVICE AGREEMENT.txt


8.2      Applicable Law. This Agreement and the rights and duties of                   Transporter and Shipper hereunder shall be governed by and                   interpreted in accordance with the laws of the State of                   Arkansas, without recourse to the law governing conflict of                   laws.
Principal Life Insurance Company - Broker Dealer Marketing and Servicing Agreement.txt


in which such person solicits a

Lets compare FAISS and BM25

In [57]:
query = "indemnification"

faiss_results = faiss_db.similarity_search(
    query,
    k=5
)

bm25_results = bm25_retriever.invoke(
    query
)
print("FAISS")

for doc in faiss_results:
    print(doc.metadata["source"])

print("\nBM25")

for doc in bm25_results:
    print(doc.metadata["source"])

FAISS
DRKOOPCOMINC_04_21_1999-EX-10.28-SPONSORSHIP AGREEMENT.txt
NICELTD_06_26_2003-EX-4.5-OUTSOURCING AGREEMENT.txt
AimmuneTherapeuticsInc_20200205_8-K_EX-10.3_11967170_EX-10.3_Development Agreement.txt
NEONSYSTEMSINC_03_01_1999-EX-10.5-DISTRIBUTOR AGREEMENT_New.txt
ALAMOGORDOFINANCIALCORP_12_16_1999-EX-1-AGENCY AGREEMENT.txt

BM25
BERKELEYLIGHTS,INC_06_26_2020-EX-10.12-COLLABORATION AGREEMENT.txt
PHREESIA,INC_05_28_2019-EX-10.18-STRATEGIC ALLIANCE AGREEMENT.txt
FUSIONPHARMACEUTICALSINC_06_05_2020-EX-10.17-Supply Agreement - FUSION.txt
UpjohnInc_20200121_10-12G_EX-2.6_11948692_EX-2.6_Manufacturing Agreement_ Supply Agreement.txt
AimmuneTherapeuticsInc_20200205_8-K_EX-10.3_11967170_EX-10.3_Development Agreement.txt


In [58]:
query = "force majeure"
faiss_results = faiss_db.similarity_search(
    query,
    k=5
)

bm25_results = bm25_retriever.invoke(
    query
)
print("FAISS")

for doc in faiss_results:
    print(doc.metadata["source"])

print("\nBM25")

for doc in bm25_results:
    print(doc.metadata["source"])

FAISS
SANDRIDGEENERGYINC_08_06_2009-EX-10.6-OPERATIONS AND MAINTENANCE AGREEMENT.txt
VERTEXENERGYINC_08_14_2014-EX-10.24-OPERATION AND MAINTENANCE AGREEMENT.txt
AimmuneTherapeuticsInc_20200205_8-K_EX-10.3_11967170_EX-10.3_Development Agreement.txt
ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.txt
PACIRA PHARMACEUTICALS, INC. - A_R STRATEGIC LICENSING, DISTRIBUTION AND MARKETING AGREEMENT .txt

BM25
PareteumCorp_20081001_8-K_EX-99.1_2654808_EX-99.1_Hosting Agreement.txt
TUNIUCORP_03_06_2014-EX-10-COOPERATION AGREEMENT.txt
BellringBrandsInc_20190920_S-1_EX-10.12_11817081_EX-10.12_Manufacturing Agreement1.txt
BIOCEPTINC_08_19_2013-EX-10-COLLABORATION AGREEMENT.txt
MIDDLEBROOKPHARMACEUTICALS,INC_03_18_2010-EX-10.1-PROMOTION AGREEMENT.txt


In [59]:
for doc in faiss_results:
    print("="*80)
    print(doc.page_content[:500])

17







ARTICLE VII  FORCE MAJEURE

7.1 Procedure.
"Force Majeure" means any cause or causes not reasonably within the control of the Party claiming suspension and which, by the exercise of reasonable diligence, such Party is unable to prevent or overcome, including, without limitation, acts of God, acts, omissions to act, and/or delays in action of federal, state, or local government or any agency thereof, strikes, lockouts, work stoppages, or other industrial disturbances, acts of a public enemy, sabotage, wars, blockades, insurrections, riots
1.32 "Force Majeure" means any circumstances whatsoever which are not within the reasonable control of the Party affected thereby, potentially including an act of God, war, act of terrorism, insurrection, riot, strike or labor dispute, shortage of materials, fire, explosion, flood, earthquake, government requisition or allocation, breakdown of or damage to plant, equipment or facilities, interruption or delay in transportation, fuel supplies 

In [60]:
for doc in bm25_results:
    print("="*80)
    print(doc.page_content[:500])

14.2 The Party prevented from fulfilling its obligations shall on becoming aware of such event inform the other Party in writing of such force majeure event as soon as possible. If the affected Party fails to inform the other Party of the occurrence of a force majeure event as set out in article 14.1 above, then such Party thereafter shall not be entitled to refer such events to force majeure as a reason for non-fulfillment. This obligation does not apply if the force majeure event is known by b
The force majeure hereunder shall mean the natural disaster, war, political event, and adjustment of laws, regulations and state policies. If the performance of this Agreement by one Party or the Parties according to provisions agreed hereunder is directly affected by the force majeure event, the affected Party shall immediately notify the other Party or its attorney-in-fact of the situation of the force majeure event, and shall, within fifteen (15) days, provide the detailed information of the

Reciprocal Rank Fusion

In [61]:
from collections import defaultdict

def reciprocal_rank_fusion(results, k=60):

    scores = defaultdict(float)
    doc_map = {}

    for docs in results:

        for rank, doc in enumerate(docs):

            key = (
                doc.metadata["source"],
                doc.page_content[:100]
            )

            scores[key] += 1/(rank+k)

            doc_map[key] = doc

    reranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [
        doc_map[key]
        for key, score in reranked
    ]

In [62]:
def hybrid_search(query, k=5):

    dense_results = faiss_db.similarity_search(
        query,
        k=k
    )

    sparse_results = bm25_retriever.invoke(
        query
    )

    fused = reciprocal_rank_fusion(
        [
            dense_results,
            sparse_results
        ]
    )

    return fused[:k]

In [63]:
query = "force majeure"

results = hybrid_search(query)

for i, doc in enumerate(results, start=1):

    print("="*100)
    print(f"RESULT {i}")

    print("\nSOURCE:")
    print(doc.metadata["source"])

    print("\nCONTRACT TYPE:")
    print(doc.metadata.get("contract_type", "Unknown"))

    print("\nTEXT:")
    print(doc.page_content[:1000])

RESULT 1

SOURCE:
SANDRIDGEENERGYINC_08_06_2009-EX-10.6-OPERATIONS AND MAINTENANCE AGREEMENT.txt

CONTRACT TYPE:
Maintenance Agreement

TEXT:
17







ARTICLE VII  FORCE MAJEURE

7.1 Procedure.
RESULT 2

SOURCE:
PareteumCorp_20081001_8-K_EX-99.1_2654808_EX-99.1_Hosting Agreement.txt

CONTRACT TYPE:
Hosting Agreement

TEXT:
14.2 The Party prevented from fulfilling its obligations shall on becoming aware of such event inform the other Party in writing of such force majeure event as soon as possible. If the affected Party fails to inform the other Party of the occurrence of a force majeure event as set out in article 14.1 above, then such Party thereafter shall not be entitled to refer such events to force majeure as a reason for non-fulfillment. This obligation does not apply if the force majeure event is known by both Parties or the affected Party is unable to inform the other Party due to the force majeure event.
RESULT 3

SOURCE:
VERTEXENERGYINC_08_14_2014-EX-10.24-OPERATION AND MAIN

In [64]:
test_queries = [
    "governing law",
    "confidentiality",
    "force majeure",
    "indemnification",
    "termination rights",
    "intellectual property"
]

for query in test_queries:

    print("\n")
    print("#"*120)
    print("QUERY:", query)
    print("#"*120)

    results = hybrid_search(query)

    for i, doc in enumerate(results, start=1):

        print("\n" + "="*100)
        print(f"RESULT {i}")

        print("\nSOURCE:")
        print(doc.metadata["source"])

        print("\nCONTRACT TYPE:")
        print(doc.metadata.get("contract_type", "Unknown"))

        print("\nTEXT:")
        print(doc.page_content[:500])



########################################################################################################################
QUERY: governing law
########################################################################################################################

RESULT 1

SOURCE:
ChinaRealEstateInformationCorp_20090929_F-1_EX-10.32_4771615_EX-10.32_Content License Agreement.txt

CONTRACT TYPE:
License Agreement

TEXT:
10.11. Governing Law. This Agreement and any dispute or claim arising out of or in connection with it or its subject matter shall be governed by,  and construed in accordance with, the laws of the People's Republic of China (without regard to its conflicts of laws rules that would mandate the  application of the laws of another jurisdiction).

RESULT 2

SOURCE:
Array BioPharma Inc. - LICENSE, DEVELOPMENT AND COMMERCIALIZATION AGREEMENT.txt

CONTRACT TYPE:
License Agreement

TEXT:
For the avoidance of doubt, the governing law set forth in Section 18.3 shall not apply to

In [65]:
test_query = "force majeure"

faiss_results = faiss_db.similarity_search(
    test_query,
    k=5
)

bm25_results = bm25_retriever.invoke(
    test_query
)

hybrid_results = hybrid_search(
    test_query,
    k=5
)

print("\nFAISS")
for doc in faiss_results:
    print(doc.metadata["source"])

print("\nBM25")
for doc in bm25_results:
    print(doc.metadata["source"])

print("\nHYBRID")
for doc in hybrid_results:
    print(doc.metadata["source"])


FAISS
SANDRIDGEENERGYINC_08_06_2009-EX-10.6-OPERATIONS AND MAINTENANCE AGREEMENT.txt
VERTEXENERGYINC_08_14_2014-EX-10.24-OPERATION AND MAINTENANCE AGREEMENT.txt
AimmuneTherapeuticsInc_20200205_8-K_EX-10.3_11967170_EX-10.3_Development Agreement.txt
ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.txt
PACIRA PHARMACEUTICALS, INC. - A_R STRATEGIC LICENSING, DISTRIBUTION AND MARKETING AGREEMENT .txt

BM25
PareteumCorp_20081001_8-K_EX-99.1_2654808_EX-99.1_Hosting Agreement.txt
TUNIUCORP_03_06_2014-EX-10-COOPERATION AGREEMENT.txt
BellringBrandsInc_20190920_S-1_EX-10.12_11817081_EX-10.12_Manufacturing Agreement1.txt
BIOCEPTINC_08_19_2013-EX-10-COLLABORATION AGREEMENT.txt
MIDDLEBROOKPHARMACEUTICALS,INC_03_18_2010-EX-10.1-PROMOTION AGREEMENT.txt

HYBRID
SANDRIDGEENERGYINC_08_06_2009-EX-10.6-OPERATIONS AND MAINTENANCE AGREEMENT.txt
PareteumCorp_20081001_8-K_EX-99.1_2654808_EX-99.1_Hosting Agreement.txt
VERTEXENERGYINC_08_14_2014-EX-10.24-OPERATION AND MAINTENANCE AGREEME

In [66]:
with open(
    VECTOR_DIR / "bm25.pkl",
    "wb"
) as f:

    pickle.dump(
        bm25_retriever,
        f
    )

print("BM25 Saved")

BM25 Saved
